In [ ]:
#lrRNAseq TAST (Total Alignment Search Tool) visualiser
#run this notebook sequentially whilst filling in required variable paths and parameters
#written in all capital letters e.g. TRANSCRIPT_TO_GENOME_BLAST_PATH

In [ ]:
#must have the following libraries installed and imported: Pandas, Numpy, Matplotlib, regex, os, Biopython, ast, operator, warnings.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import matplotlib.patches as patches
import matplotlib
from Bio import SeqIO
from matplotlib.ticker import MaxNLocator
from ast import literal_eval
from operator import itemgetter
import warnings
warnings.filterwarnings("ignore")
%run "./bioinformatic_functions.ipynb" #import functions from the "bioinformatic_functions" file.

In [ ]:
#Align long read RNAseq transcripts to a genomic region using BLAST with alignment parameters (-word_size, -evalue) of your choice.
#BLAST output must be -outfmt '6 qseqid sseqid slen qlen qstart qend sstart send length mismatch gapopen pident evalue bitscore' (no header line)

#import lrRNAseq BLAST alignments to genomic region
TRANSCRIPT_TO_GENOME_BLAST_PATH = ""
#e.g. TRANSCRIPT_TO_GENOME_BLAST_PATH = "./lrRNAseq_to_Ef318A10.out"
all_blast = read_blast(TRANSCRIPT_TO_GENOME_BLAST_PATH)

In [ ]:
#choose which alignment strand(s) to keep for each transcript
#"dominant" - keep only the strand with the highest total bitscore for each transcript (recommended; avoids merging sense and antisense hits into one model)
#"both" - keep alignments on both strands (behaviour of version 1.0)
#"plus" / "minus" - keep only alignments where the genome is hit in forward / reverse orientation
STRAND_MODE = "dominant"

if STRAND_MODE == "dominant":
    strand_score = all_blast.groupby(["qseqid", "sorientation"])["bitscore"].sum().unstack(fill_value=0)
    if "+" not in strand_score: strand_score["+"] = 0
    if "-" not in strand_score: strand_score["-"] = 0
    dominant = (strand_score["-"] > strand_score["+"]).map({True: "-", False: "+"}) #ties go to "+"
    all_blast = all_blast[all_blast["sorientation"] == all_blast["qseqid"].map(dominant)]
elif STRAND_MODE == "plus":
    all_blast = all_blast[all_blast["sorientation"] == "+"]
elif STRAND_MODE == "minus":
    all_blast = all_blast[all_blast["sorientation"] == "-"]
elif STRAND_MODE != "both":
    raise ValueError('STRAND_MODE must be "dominant", "both", "plus" or "minus"')
all_blast = all_blast.reset_index(drop=True)
print(f"{all_blast.shape[0]} alignments from {all_blast['qseqid'].nunique()} transcripts kept with STRAND_MODE = {STRAND_MODE}")

In [ ]:
#calculate transcript filtration variables i.e. transcript coverage and maximal intron/alignment gap length
#merges overlapping alignments of each transcript (on the transcript for coverage, on the genome for gaps between alignment blocks)
merged_q = merge_intervals(all_blast, "qstart", "qend")
cov_dict = (100 * (merged_q["Right"] - merged_q["Left"]).groupby(merged_q["qseqid"]).sum()
            / all_blast.groupby("qseqid")["qlen"].max()).to_dict()

merged_s = merge_intervals(all_blast, "sstart", "send")
merged_s["gap"] = merged_s["Left"] - merged_s.groupby("qseqid")["Right"].shift()
dist_dict = merged_s.dropna(subset="gap").groupby("qseqid")["gap"].apply(lambda x: x.astype(int).tolist()).to_dict()

all_blast["Tr_coverage"] = all_blast["qseqid"].map(cov_dict)
all_blast["s_distances"] = all_blast["qseqid"].apply(lambda x: dist_dict.get(x, [0]))
all_blast["max_distance"] = all_blast["s_distances"].apply(max)
all_blast["srange"] = [[s, e] for s, e in zip(all_blast["sstart"].tolist(), all_blast["send"].tolist())]
all_blast["qrange"] = [[s, e] for s, e in zip(all_blast["qstart"].tolist(), all_blast["qend"].tolist())]
all_blast["counts"] = all_blast.groupby("qseqid")["sseqid"].transform("count")

In [ ]:
cov50=all_blast.sort_values(["sstart", "qstart"], kind="stable").reset_index(drop=True)

In [ ]:
#filter by maximal intron length
MAX_INTRON_LENGTH = 20000 #bp
cov50 = cov50[cov50["max_distance"]<MAX_INTRON_LENGTH]

In [ ]:
#filter by minimal percentage of transcript length covered by the genome as per collective BLAST alignments
MIN_COVERAGE = 25 #percent
cov50 = cov50[cov50["Tr_coverage"]>MIN_COVERAGE]

In [ ]:
#save filtered transcript ids to list in main directory
FILTERED_TRANSCRIPT_IDS = ""
#e.g. FILTERED_TRANSCRIPT_IDS = "./aligned_lrRNAseq_cov25_filtered.txt"
write_list(cov50.drop_duplicates("qseqid")["qseqid"].tolist(), FILTERED_TRANSCRIPT_IDS)

In [ ]:
#produce a FASTA file including only the transcripts that passed the filtration procedure
ALL_TRANSCRIPTS_PATH = "" # enter path to the bulk RNAseq file
#e.g. ALL_TRANSCRIPTS_PATH = "./all_transcripts.fasta"
FILTERED_TRANSCRIPTS_FASTA_PATH = "" #filtered transcripts FASTA path
#e.g. FILTERED_TRANSCRIPTS_FASTA_PATH = "./filtered_transcripts.fasta"
if os.path.abspath(ALL_TRANSCRIPTS_PATH) == os.path.abspath(FILTERED_TRANSCRIPTS_FASTA_PATH):
    raise ValueError("FILTERED_TRANSCRIPTS_FASTA_PATH must be different from ALL_TRANSCRIPTS_PATH, otherwise the input file would be overwritten")
keep_ids = set(cov50["qseqid"])
n_written = SeqIO.write((rec for rec in SeqIO.parse(ALL_TRANSCRIPTS_PATH, "fasta") if rec.id in keep_ids),
                        FILTERED_TRANSCRIPTS_FASTA_PATH, "fasta")
print(f"{n_written} of {len(keep_ids)} filtered transcripts written to {FILTERED_TRANSCRIPTS_FASTA_PATH}")
if n_written < len(keep_ids):
    print("WARNING: some filtered transcript IDs were not found in ALL_TRANSCRIPTS_PATH")

In [ ]:
#import filtered transcripts FASTA and find ORF regions/stats
BARCODE_LEN = 0 #trim n barcode characters from both ends of transcripts #set to 39 if working with data provided in the example
cov50_fasta = parse_fasta(FILTERED_TRANSCRIPTS_FASTA_PATH,rm_barcode=BARCODE_LEN)[0]
all_stats = find_tr_ORF_stats(cov50_fasta["seq"].tolist(),cov50_fasta["id"].tolist(),tissue=True, include_seq=True)
if all_stats.shape[0] == 0:
    raise ValueError("No ORFs were found in any filtered transcript")

#ORFs are found on barcode-trimmed sequences, but BLAST coordinates (qstart/qend) refer to the sequences that were BLASTed.
#if BLAST was run on untrimmed sequences, ORF coordinates are shifted by BARCODE_LEN so that both use the same numbering.
blast_qlen = all_blast.groupby("qseqid")["qlen"].max()
trimmed_len = cov50_fasta.set_index("id")["seq"].str.len()
orf_offset = ((blast_qlen.reindex(trimmed_len.index) - trimmed_len) // 2).fillna(0).astype(int)
if not orf_offset.isin([0, BARCODE_LEN]).all():
    print("WARNING: transcript lengths in the BLAST file and FASTA file differ by something other than the barcode length for:",
          orf_offset[~orf_offset.isin([0, BARCODE_LEN])].index.tolist())
if (orf_offset != 0).any():
    print(f"ORF coordinates of {(orf_offset != 0).sum()} transcripts shifted by the trimmed barcode length to match BLAST coordinates")

In [ ]:
#import FASTA of the genomic region of interest e.g. BAC
GENOME_FASTA_PATH = ""
#e.g. GENOME_FASTA_PATH = "./Ef318A10.fst"
my_seq = "".join([line.strip() for line in open(GENOME_FASTA_PATH) if not line.startswith(">")])
seq_len = len(my_seq)

In [ ]:
#calculate ORF overlap statistics
all_s = all_stats[["id", "ORF_coords", "all_ORF_coords", "ORF_lengths", "num_ORFs_found"]].rename(columns={"id":"qseqid"})
all_s["ORF_coords"] = [[c + orf_offset.get(i, 0) for c in coords] for i, coords in zip(all_s["qseqid"], all_s["ORF_coords"])]
all_s["all_ORF_coords"] = [[[c + orf_offset.get(i, 0) for c in orf] for orf in orfs] for i, orfs in zip(all_s["qseqid"], all_s["all_ORF_coords"])]

cov50_ranges = cov50.groupby("qseqid").agg({"srange": "sum", "qrange": "sum"})
cov50_ranges = cov50_ranges.reset_index()

cov50_ranges["start"] = cov50_ranges["srange"].apply(lambda x: min(x))
cov50_ranges["end"] = cov50_ranges["srange"].apply(lambda x: max(x))

cov50_ranges["qstarts"] = cov50_ranges["qrange"].apply(lambda x: x[::2])
cov50_ranges["TrRange"] = cov50_ranges.apply(lambda x: [x["start"], x["end"]], axis=1)
cov50_ranges["TrLen"] = cov50_ranges["end"] - cov50_ranges["start"]

no_ORF_ids = sorted(set(cov50_ranges["qseqid"]) - set(all_s["qseqid"]))
if no_ORF_ids:
    print(f"{len(no_ORF_ids)} transcripts have no ORF and will not be plotted:", no_ORF_ids)
cov50_ranges = cov50_ranges.merge(right=all_s, on="qseqid",how="inner")
cov50_ranges["left_ORF_coords"] = cov50_ranges["all_ORF_coords"].apply(lambda x: [i[0] for i in x])
cov50_ranges["right_ORF_coords"] = cov50_ranges["all_ORF_coords"].apply(lambda x: [i[1] for i in x])

cov50_ranges["overlap_ORF_coords"] = cov50_ranges.apply(lambda x: calc_overlaps(pd.DataFrame({"Left":x.left_ORF_coords,"Right":x.right_ORF_coords}),by="LeftRight")[0]["Coords"].tolist(), axis=1)

cov50_ranges["qranges"] = cov50_ranges["qrange"].apply(lambda x: [[[x[::2], x[1::2]][0][i], [x[::2], x[1::2]][1][i]] for i in range(len([x[::2], x[1::2]][0]))])

cov50_ranges["in_ORF_matrix"] = cov50_ranges.apply(lambda x: [bool(list(pd.RangeIndex(i[0],i[1]).intersection(pd.RangeIndex(b[0],b[1])))) for i in x.qranges for b in x.overlap_ORF_coords], axis=1)
cov50_ranges["in_ORF_matrix_largest"] = cov50_ranges.apply(lambda x: [bool(list(pd.RangeIndex(i[0],i[1]).intersection(pd.RangeIndex(b[0],b[1])))) for i in x.qranges for b in [x.ORF_coords]], axis=1)

cov50_ranges["qranges_in_ORF"] = cov50_ranges.apply(lambda x: [any(z) for z in [[bool(list(pd.RangeIndex(i[0],i[1]).intersection(pd.RangeIndex(b[0],b[1])))) for i in x.qranges for b in x.overlap_ORF_coords][v:v+len(x.overlap_ORF_coords)] for v in range(0,len(x.in_ORF_matrix),len(x.overlap_ORF_coords))]], axis=1)
cov50_ranges["qranges_in_largest_ORF"] = cov50_ranges.apply(lambda x: [any(z) for z in [[bool(list(pd.RangeIndex(i[0],i[1]).intersection(pd.RangeIndex(b[0],b[1])))) for i in x.qranges for b in [x.ORF_coords]][v:v+len([x.ORF_coords])] for v in range(0,len(x.in_ORF_matrix_largest),len([x.ORF_coords]))]], axis=1)

cov50_ranges["qstarts_sorted_values"] = cov50_ranges["qstarts"].apply(lambda x: sort_with_order(x, make_equal=False)[0])
cov50_ranges["qstarts_sorted_order"] = cov50_ranges["qstarts"].apply(lambda x: sort_with_order(x, make_equal=False)[1])
cov50_ranges["qstarts_sorted_order_sequential"] = cov50_ranges["qstarts"].apply(lambda x: sort_with_order(x, make_sequential=True)[1])#true order of duplicate is lost

In [ ]:
#arrange transcripts into plot rows: transcripts that do not overlap on the genome share a row
#greedy interval partitioning - transcripts are taken in order of genomic start and placed in the row that became free earliest,
#which gives the smallest possible number of rows
MIN_GAP = 0 #bp, minimal genomic distance between neighbouring transcripts sharing a row
cov50_ranges_ = cov50_ranges.sort_values(by=["start", "end"]).reset_index(drop=True)
cov50_ranges_["NumExon"] = cov50_ranges_["srange"].apply(lambda x: len(x)/2)
cov50_ranges_["index"] = cov50_ranges_.index

import heapq
row_members = [] #list of row index lists
free_rows = [] #heap of (end of last transcript in row, row number)
for i, (start, end) in enumerate(zip(cov50_ranges_["start"], cov50_ranges_["end"])):
    if free_rows and free_rows[0][0] + MIN_GAP < start:
        _, row = heapq.heappop(free_rows)
        row_members[row].append(i)
    else:
        row = len(row_members)
        row_members.append([i])
    heapq.heappush(free_rows, (end, row))

lrRNAseq_plot_df = pd.DataFrame({
    "coords": [[c for i in m for c in cov50_ranges_["srange"][i]] for m in row_members],
    "total_exons": [sum(cov50_ranges_["NumExon"][i] for i in m) for m in row_members],
    "TrNames": [[cov50_ranges_["qseqid"][i] for i in m] for m in row_members],
    "exons_per_tr": [[cov50_ranges_["NumExon"][i] for i in m] for m in row_members],
    "range_per_tr": [[cov50_ranges_["TrRange"][i] for i in m] for m in row_members],
    "Total_aln_len": [sum(cov50_ranges_["TrLen"][i] for i in m) for m in row_members],
})
lrRNAseq_plot_df = lrRNAseq_plot_df.sort_values("Total_aln_len", ascending=False).reset_index(drop=True)
print(f"{cov50_ranges_.shape[0]} transcripts arranged in {lrRNAseq_plot_df.shape[0]} rows")

In [ ]:
#combine transcript plot dataframe with the filtered BLAST dataframe
lrRNAseq_plot_df["ORF_coords"] = ""
lrRNAseq_plot_df["all_ORF_coords"] = ""
lrRNAseq_plot_df["qranges_in_largest_ORF"] = ""
lrRNAseq_plot_df["qranges_in_ORF"] = ""
lrRNAseq_plot_df["qstarts_sorted_values"] = ""
lrRNAseq_plot_df["qstarts_sorted_order"] = ""
lrRNAseq_plot_df["qstarts_sorted_order_sequential"] = ""
lrRNAseq_plot_df["qstarts"] = ""
lrRNAseq_plot_df["index"] = lrRNAseq_plot_df.index

for idx, row in lrRNAseq_plot_df.iterrows():
    qstarts_sorted_order_sequential_list = []
    qstarts_sorted_order_list = []
    qstarts_sorted_values_list = []
    qranges_in_largest_ORF_list = []
    qranges_in_ORF_list = []
    all_ORF_coords_list = []
    ORF_coords_list = []
    qstarts_list = []
    if type(lrRNAseq_plot_df["TrNames"][idx])==str:
        id = lrRNAseq_plot_df["TrNames"][idx]
        qstarts_sorted_order_sequential_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts_sorted_order_sequential"].tolist()[0])
        qstarts_sorted_order_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts_sorted_order"].tolist()[0])
        qstarts_sorted_values_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts_sorted_values"].tolist()[0])
        qranges_in_largest_ORF_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qranges_in_largest_ORF"].tolist()[0])
        qranges_in_ORF_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qranges_in_ORF"].tolist()[0])
        all_ORF_coords_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["all_ORF_coords"].tolist()[0])
        ORF_coords_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["ORF_coords"].tolist()[0])
        qstarts_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts"].tolist()[0])
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "ORF_coords"] = str(ORF_coords_list)
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "all_ORF_coords"] = str(all_ORF_coords_list)
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qranges_in_largest_ORF"] = str(qranges_in_largest_ORF_list)
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qranges_in_ORF"] = str(qranges_in_ORF_list)
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts_sorted_values"] = str(qstarts_sorted_values_list)
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts_sorted_order"] = str(qstarts_sorted_order_list)
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts_sorted_order_sequential"] = str(qstarts_sorted_order_sequential_list)
        lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts"] = str(qstarts_list)
    else:    
        for id in lrRNAseq_plot_df["TrNames"][idx]:
            qstarts_sorted_order_sequential_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts_sorted_order_sequential"].tolist()[0])
            qstarts_sorted_order_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts_sorted_order"].tolist()[0])
            qstarts_sorted_values_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts_sorted_values"].tolist()[0])
            qranges_in_largest_ORF_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qranges_in_largest_ORF"].tolist()[0])
            qranges_in_ORF_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qranges_in_ORF"].tolist()[0])
            all_ORF_coords_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["all_ORF_coords"].tolist()[0])
            ORF_coords_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["ORF_coords"].tolist()[0])
            qstarts_list.append(cov50_ranges_[cov50_ranges_["qseqid"]==id]["qstarts"].tolist()[0])
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "ORF_coords"] = str(ORF_coords_list)
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "all_ORF_coords"] = str(all_ORF_coords_list)
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qranges_in_largest_ORF"] = str(qranges_in_largest_ORF_list)
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qranges_in_ORF"] = str(qranges_in_ORF_list)
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts_sorted_values"] = str(qstarts_sorted_values_list)
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts_sorted_order"] = str(qstarts_sorted_order_list)
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts_sorted_order_sequential"] = str(qstarts_sorted_order_sequential_list)
            lrRNAseq_plot_df.loc[lrRNAseq_plot_df["index"]== idx, "qstarts"] = str(qstarts_list)

lrRNAseq_plot_df["ORF_coords"] = lrRNAseq_plot_df["ORF_coords"].apply(lambda x: eval(x))
lrRNAseq_plot_df["all_ORF_coords"] = lrRNAseq_plot_df["all_ORF_coords"].apply(lambda x: eval(x))
lrRNAseq_plot_df["qranges_in_largest_ORF"] = lrRNAseq_plot_df["qranges_in_largest_ORF"].apply(lambda x: eval(x))
lrRNAseq_plot_df["qranges_in_ORF"] = lrRNAseq_plot_df["qranges_in_ORF"].apply(lambda x: eval(x))
lrRNAseq_plot_df["qstarts_sorted_values"] = lrRNAseq_plot_df["qstarts_sorted_values"].apply(lambda x: eval(x))
lrRNAseq_plot_df["qstarts_sorted_order"] = lrRNAseq_plot_df["qstarts_sorted_order"].apply(lambda x: eval(x))
lrRNAseq_plot_df["qstarts_sorted_order_sequential"] = lrRNAseq_plot_df["qstarts_sorted_order_sequential"].apply(lambda x: eval(x))
lrRNAseq_plot_df["qstarts"] = lrRNAseq_plot_df["qstarts"].apply(lambda x: eval(x))

In [ ]:
#plotting requirements
lrRNAseq_plot_df = lrRNAseq_plot_df.sort_values(by="Total_aln_len", ascending=False)
lrRNAseq_plot_df["coverage"] = lrRNAseq_plot_df["total_exons"].apply(lambda x: [0]+[0,1,1,0]*int(x)+[0] )

In [ ]:
#further processing of plotting coordinates
lrRNAseq_plot_df["coords2"] = lrRNAseq_plot_df["coords"].apply(lambda c: [0] + c + [x - 1 for x in c[::2]] + [x + 1 for x in c[1::2]] + [len(my_seq)])

In [ ]:
#plotting coordinate statistics
lrRNAseq_plot_df["min"] = lrRNAseq_plot_df["coords"].apply(lambda x: min(x))
lrRNAseq_plot_df["max"] = lrRNAseq_plot_df["coords"].apply(lambda x: max(x))
lrRNAseq_plot_df["span"] = lrRNAseq_plot_df["max"] - lrRNAseq_plot_df["min"]
lrRNAseq_plot_df.sort_values(by="span", ascending=False, inplace=True)

In [ ]:
#translate order and get coord ranges
lrRNAseq_plot_df["qstarts_sorted_order_translated"] = lrRNAseq_plot_df["qstarts_sorted_order_sequential"].apply(lambda x: [translate_order(i) for i in x])
lrRNAseq_plot_df["coord_ranges"] = lrRNAseq_plot_df["coords"].apply(lambda x: [[[x[::2],x[1::2]][0][i],[x[::2],x[1::2]][1][i]]for i in range(len([x[::2],x[1::2]][0]))])

In [ ]:
#create order dictionary for each transcript
dict_list_outer = []
for idx, row in lrRNAseq_plot_df.iterrows():
    dict_list = []
    for i in range(len(lrRNAseq_plot_df["qstarts_sorted_order_sequential"][idx])):        
        d = {}
        for j in range(len(lrRNAseq_plot_df["qstarts_sorted_order_sequential"][idx][i])):
            d[lrRNAseq_plot_df["qstarts_sorted_values"][idx][i][j] ] = lrRNAseq_plot_df["qstarts_sorted_order_sequential"][idx][i][j]
        dict_list.append(d)
    dict_list_outer.append(dict_list)
lrRNAseq_plot_df["order_dict"] = dict_list_outer

In [ ]:
#use order dictionary to check order of alignment regions in transcripts
ordered_list_outer = []
for idx, row in lrRNAseq_plot_df.iterrows():
    ordered_list = []
    for j in range(len(lrRNAseq_plot_df["order_dict"][idx])):
        
        o_dict = lrRNAseq_plot_df["order_dict"][idx][j]
        qstarts = lrRNAseq_plot_df["qstarts"][idx][j]
        ordered = []
        
        for i in range(len(qstarts)):
            ordered.append(o_dict[qstarts[i]])
        ordered_list.append(ordered)
    ordered_list_outer.append(ordered_list)
lrRNAseq_plot_df["exon_synteny"] = ordered_list_outer
lrRNAseq_plot_df["centre_coords"] = lrRNAseq_plot_df["coord_ranges"].apply(lambda x: [int(i[0]+((i[1]-i[0])/2)) for i in x])

In [ ]:
#ordered exon indexes and correction for closely bundled centre coordinates for better visibility of the numbers in the plot
lrRNAseq_plot_df["syntenous_indexes"] = lrRNAseq_plot_df["exon_synteny"].apply(lambda x: [syntenous_indexes(i) for i in x])
lrRNAseq_plot_df["centre_coords_corrected"] = lrRNAseq_plot_df["centre_coords"].apply(lambda x: space_elements(x))

In [ ]:
#required for plotting multi-transcript syntentic regions
#convert exons_per_tr to lists
exons_per_tr_list = []
for idx, row in lrRNAseq_plot_df.iterrows():
    ex_per_tr = lrRNAseq_plot_df["exons_per_tr"][idx]
    if type(ex_per_tr) == float:
        exons_per_tr_list.append([int(ex_per_tr)])
    else:
        exons_per_tr_list.append(ex_per_tr)
lrRNAseq_plot_df["exons_per_tr"] = exons_per_tr_list


In [ ]:
#make syntenous_indexes lists have even number of members
new_xs = []
for idx, row in lrRNAseq_plot_df.iterrows():
    x = lrRNAseq_plot_df["syntenous_indexes"][idx].copy()
# x = [[0,3,0],[0],[0,4,5,7]]
    new_x = []
    for i in x:
        if len(i)%2!=0:
            i.append(i[len(i)-1])
        new_x.append(i)
    new_xs.append(new_x)
lrRNAseq_plot_df["syntenous_indexes_even"] = new_xs

In [ ]:
#increment subsequent transcript lists by number of exons in previous transcripts
syn_idxs_corr_list = []
for idx, row in lrRNAseq_plot_df.iterrows():
    syn_idxs = lrRNAseq_plot_df["syntenous_indexes_even"][idx].copy()
    # syn_idxs = [[0, 1], [0, 5, 6, 7, 8, 14, 15, 16, 22],[0,3]]
    ex_per_tr = lrRNAseq_plot_df["exons_per_tr"][idx].copy()
    # ex_per_tr = [2.0, 23.0, 4]
    num_trs = len(syn_idxs)
    increment = 0
    syn_idxs_corrected = []
    syn_idxs_corrected.append(syn_idxs[0])
    
    for i in range(1,num_trs):
        increment += ex_per_tr[i-1]
        syn_idxs_corrected.append((np.array(syn_idxs[i])+increment).tolist())
    syn_idxs_corr_list.append(syn_idxs_corrected)
lrRNAseq_plot_df["syn_idxs_corrected"] = syn_idxs_corr_list

In [ ]:
#finalising information
lrRNAseq_plot_df["qranges_in_ORF_unwrapped"] = lrRNAseq_plot_df["qranges_in_ORF"].apply(lambda x: [i for i in extract_nested_values(x)] )
lrRNAseq_plot_df["qranges_in_largest_ORF_unwrapped"] = lrRNAseq_plot_df["qranges_in_largest_ORF"].apply(lambda x: [i for i in extract_nested_values(x)] )
lrRNAseq_plot_df["exon_synteny_unwrapped"] = lrRNAseq_plot_df["exon_synteny"].apply(lambda x: [i for i in extract_nested_values(x)] )
lrRNAseq_plot_df["syntenous_indexes_unwrapped"] = lrRNAseq_plot_df["syn_idxs_corrected"].apply(lambda x: [i for i in extract_nested_values(x)] )
lrRNAseq_plot_df["syntenous_indexes_int_unwrapped"]  = lrRNAseq_plot_df["syntenous_indexes_unwrapped"].apply(lambda x: [int(i) for i in x])

In [ ]:
#plot results
seq_length = seq_len
matplotlib.rcParams["agg.path.chunksize"] = 10000
plt.rcParams['agg.path.chunksize'] = 10000
figsize_param_w = min(int(seq_length/1100), 50)
figsize_param_h = max(int(lrRNAseq_plot_df.shape[0]/2.5), 2)
plt.rcParams['figure.figsize'] = [figsize_param_w, figsize_param_h]
plt.rcParams['figure.dpi'] = 140

nrows = lrRNAseq_plot_df.shape[0]
fig, axes = plt.subplots(nrows=nrows, ncols=1, sharex=False, squeeze=False)
axes = axes[:, 0]
plt.subplots_adjust(top=0.9, bottom=0.1, hspace=0)
axes[lrRNAseq_plot_df.shape[0]-1].set_xlabel("")


lrRNAseq_plot_df.reset_index(drop=True, inplace=True)

for idx, row in lrRNAseq_plot_df.iterrows():
#plot all alignment regions
    temp_plot_df = pd.DataFrame()
    temp_plot_df["coords"] = lrRNAseq_plot_df["coords2"][idx]
    temp_plot_df.sort_values(by="coords", inplace=True)
    temp_plot_df["coverage"] = lrRNAseq_plot_df["coverage"][idx]
    temp_plot_df.plot.area(x="coords", y= "coverage", figsize=(figsize_param_w, figsize_param_h), grid=True,
                           legend=False, ax = axes[idx], color='r', linewidth=1, xlim=[0,seq_length],zorder=1)
#plot line if only 1 line in lane
    temp_lines = pd.DataFrame()
    if type(lrRNAseq_plot_df["range_per_tr"][idx][0]) == int:
        temp_lines["coord"] = lrRNAseq_plot_df["range_per_tr"][idx]
        temp_lines["midline"] = 0.5
        temp_lines.plot.line(x="coord", y= "midline", figsize=(figsize_param_w, figsize_param_h), grid=True, 
                             legend=False, ax = axes[idx], color='b', linewidth=3, xlim=[0,seq_length])
#plot line if multiple lines in lane
    else:
        c = ["b","g","m","c","w","grey","k", "silver", "beige","teal","lightskyblue","r","y","lightgreen","peru"]
        i=0
        for line in lrRNAseq_plot_df["range_per_tr"][idx]:
            temp_lines["coord"] = line
            temp_lines["midline"] = 0.5
            temp_lines.plot.line(x="coord", y= "midline", figsize=(figsize_param_w, figsize_param_h), grid=True,
                                   legend=False, ax = axes[idx], color=c[i % len(c)], linewidth=4, xlim=[0,seq_length])
            i+=1
#plot alignment region in ORF as a rectangle patch - border only
    v = 0
    coord_ranges = lrRNAseq_plot_df["coord_ranges"][idx]
    qranges_in_ORF = lrRNAseq_plot_df["qranges_in_ORF_unwrapped"][idx]
    # qranges_in_ORF = lrRNAseq_plot_df["qranges_in_ORF"][idx]
    for c in coord_ranges:
        if qranges_in_ORF[v] == True:
            a2 = patches.Rectangle((c[0],0),c[1]-c[0],1,color="gold", fill=None,zorder=1)
            axes[idx].add_patch(a2)
        v+=1
#plot alignment region in largest ORF as a rectangle patch - border and fill
    v = 0
    qranges_in_largest_ORF = lrRNAseq_plot_df["qranges_in_largest_ORF_unwrapped"][idx]
    for c in coord_ranges:
        if qranges_in_largest_ORF[v] == True:
            a1 = patches.Rectangle((c[0],0),c[1]-c[0],1,color="gold",zorder=99)
            a2 = patches.Rectangle((c[0],0),c[1]-c[0],1,color="gold", fill=None,zorder=1)
            axes[idx].add_patch(a1)
            axes[idx].add_patch(a2)
        v+=1

#plot text of syntentic order 
    centre_coords = lrRNAseq_plot_df["centre_coords"][idx]
    syntentic_order = lrRNAseq_plot_df["exon_synteny_unwrapped"][idx]
    for i in range(len(centre_coords)):
        fig.text(centre_coords[i], int((nrows-idx)/(nrows+1))+0.5, f"{syntentic_order[i]}", horizontalalignment="center", transform=axes[idx].transData, verticalalignment="center", fontsize=15, color="black")

#plot syntentic frame
    v = 0
    coord_ranges = lrRNAseq_plot_df["coord_ranges"][idx]
    structure = lrRNAseq_plot_df["syntenous_indexes_int_unwrapped"][idx]
    syntentic_coords = []
    for i in range(0,len(structure)-1,2):
        start = coord_ranges[structure[i]][0]
        end = coord_ranges[structure[i+1]][1]
        syntentic_coords.append([start,end])
    for c in syntentic_coords:
        # a4 = patches.Rectangle((c[0],0)c[1]-c[0],1,color="pink",zorder=1)
        a3 = patches.Rectangle((c[0],0),c[1]-c[0],1,color="pink",zorder=0)
        axes[idx].add_patch(a3)
        # axes[idx].add_patch(a4)
        v+=1

for idx in range(nrows):
    if idx != 0 and idx != nrows-1:
        axes[idx].set_xticklabels([])
        axes[idx].set_xlabel("")
        axes[idx].minorticks_on()
        
        # axes[idx].tick_params(axis='both', which='major', labelsize=30, pad=15, width=3, size=8) #labelbottom=True,labeltop=True,bottom=True,top=True
        # axes[idx].tick_params(axis='both', which='minor', labelsize=30, pad=15, width=2, size=5) #labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='y', which='major', labelsize=30, pad=0, width=0, size=0) #labelbottom=True,labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='y', which='minor', labelsize=0, pad=0, width=0, size=0) #labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='x', which='major', labelsize=30, pad=15, width=3, size=8,labelbottom=True,labeltop=True,bottom=True,top=True) #labelbottom=True,labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='x', which='minor', labelsize=30, pad=15, width=2, size=5,labelbottom=True,labeltop=True,bottom=True,top=True) #labeltop=True,bottom=True,top=True
        axes[idx].grid(which="major", axis="x", color = 'black', linestyle = '--', dashes=(5, 5), linewidth = 1.35, alpha=0.65) 
        axes[idx].grid(which="minor", axis="x", color = 'black', linestyle = '-', dashes=(4, 5), linewidth = 0.85, alpha=0.45)    
        for axis in ["top", "bottom", "left", "right"]: 
            axes[idx].spines[axis].set_linewidth(2.5)
        axes[0].xaxis.tick_top()
        
    elif idx == 0:
        axes[idx].set_xlabel("")
        axes[idx].minorticks_on()
        
        # axes[idx].tick_params(axis='both', which='major', labelsize=30, pad=15, width=3, size=8) #labelbottom=True,labeltop=True,bottom=True,top=True
        # axes[idx].tick_params(axis='both', which='minor', labelsize=30, pad=15, width=2, size=5) #labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='y', which='major', labelsize=30, pad=0, width=0, size=0) #labelbottom=True,labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='y', which='minor', labelsize=0, pad=0, width=0, size=0) #labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='x', which='major', labelsize=30, pad=15, width=3, size=8,labelbottom=0,labeltop=True,bottom=0,top=True) #labelbottom=True,labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='x', which='minor', labelsize=30, pad=15, width=2, size=5,labelbottom=0,labeltop=True,bottom=0,top=True) #labeltop=True,bottom=True,top=True
        axes[idx].grid(which="major", axis="x", color = 'black', linestyle = '--', dashes=(5, 5), linewidth = 1.35, alpha=0.65) 
        axes[idx].grid(which="minor", axis="x", color = 'black', linestyle = '-', dashes=(4, 5), linewidth = 0.85, alpha=0.45)    
        for axis in ["top", "bottom", "left", "right"]: 
            axes[idx].spines[axis].set_linewidth(2.5)
        axes[0].xaxis.tick_top()

    elif idx == nrows-1:
        # axes[idx].tick_params(axis='both', which='major', labelsize=30, pad=15, width=3, size=8) #labelbottom=True,labeltop=True,bottom=True,top=True
        # axes[idx].tick_params(axis='both', which='minor', labelsize=30, pad=15, width=2, size=5) #labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='y', which='major', labelsize=30, pad=0, width=0, size=0) #labelbottom=True,labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='y', which='minor', labelsize=0, pad=0, width=0, size=0) #labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='x', which='major', labelsize=30, pad=15, width=3, size=8,labelbottom=True,labeltop=0,bottom=True,top=0) #labelbottom=True,labeltop=True,bottom=True,top=True
        axes[idx].tick_params(axis='x', which='minor', labelsize=30, pad=15, width=2, size=5,labelbottom=True,labeltop=0,bottom=True,top=0) #labeltop=True,bottom=True,top=True
        axes[idx].grid(which="major", axis="x", color = 'black', linestyle = '--', dashes=(5, 5), linewidth = 1.35, alpha=0.65) 
        axes[idx].grid(which="minor", axis="x", color = 'black', linestyle = '-', dashes=(4, 5), linewidth = 0.85, alpha=0.45)    
        for axis in ["top", "bottom", "left", "right"]: 
            axes[idx].spines[axis].set_linewidth(2.5)
        axes[0].xaxis.tick_top()
        axes[idx].set_xlabel("")
        axes[idx].minorticks_on()
        
    for z, label in enumerate(axes[idx].get_yticklabels()):
        if z >= 0 and z < len(axes[idx].get_yticklabels()):
            label.set_visible(False)
    
    axes[idx].set_yticklabels([])

In [ ]:
#the rows of plot are indexed by the index of lrRNAseq_plot_df dataframe
#TrNames column of lrRNAseq_plot_df contains the names of transcript for the corresponding row of the plot